Lets stitch together the MTBS dataset:
most recent burn year
time since most recent burn
burn count
ever burned?
most recent severity

In [4]:
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio
import xarray as xr


# ============================================================
# USER PATHS
# ============================================================
tif_path = Path(r"D:\My Drive\GEE_exports\MTBS_BurnContext_Multiband_10m.tif")
schema_path = Path(r"D:\My Drive\GEE_exports\MTBS_BurnContext_Schema.csv")
out_nc = tif_path.with_name("MTBS_BurnContext_Multiband_10m_finalized.nc")


# ============================================================
# HELPERS
# ============================================================
def read_schema(schema_path: Path) -> pd.DataFrame:
    """
    Read the GEE-exported schema CSV and retain only valid band rows.

    Handles:
      - no metadata_key / metadata_value columns
      - trailing blank rows
      - extra columns like system:index and .geo
    """
    schema = pd.read_csv(schema_path)
    schema.columns = [str(c).strip() for c in schema.columns]

    required = ["band_index", "band_name", "description", "dtype", "nodata", "units"]
    missing = [c for c in required if c not in schema.columns]
    if missing:
        raise ValueError(f"Schema is missing required columns: {missing}")

    # Keep only rows that actually define raster bands
    band_schema = schema[schema["band_name"].notna()].copy()

    if band_schema.empty:
        raise ValueError("No valid band rows found in schema CSV.")

    # Drop rows with empty band names just in case
    band_schema["band_name"] = band_schema["band_name"].astype(str).str.strip()
    band_schema = band_schema[band_schema["band_name"] != ""].copy()

    # Standardize band index
    band_schema["band_index"] = pd.to_numeric(
        band_schema["band_index"], errors="coerce"
    )
    band_schema = band_schema[band_schema["band_index"].notna()].copy()
    band_schema["band_index"] = band_schema["band_index"].astype(int)

    band_schema = band_schema.sort_values("band_index").reset_index(drop=True)
    return band_schema


def maybe_numeric(value):
    if pd.isna(value):
        return None
    try:
        f = float(value)
        if f.is_integer():
            return int(f)
        return f
    except Exception:
        return value


def dtype_from_schema(dtype_str: str):
    """
    Map schema dtype strings to numpy dtypes for NetCDF encoding.
    """
    s = str(dtype_str).strip().lower()
    if s == "byte":
        return np.int16   # your exported TIFF stores all bands as int16
    if s == "int16":
        return np.int16
    if s == "uint16":
        return np.uint16
    if s == "int32":
        return np.int32
    if s == "float":
        return np.float32
    if s == "double":
        return np.float64
    raise ValueError(f"Unsupported schema dtype: {dtype_str}")


# ============================================================
# READ SCHEMA
# ============================================================
band_schema = read_schema(schema_path)

print("\nSchema columns:")
print(pd.read_csv(schema_path, nrows=1).columns.tolist())

print("\nBand schema:")
print(band_schema[["band_index", "band_name", "dtype", "units", "nodata"]])


# ============================================================
# READ GEOTIFF
# ============================================================
with rasterio.open(tif_path) as src:
    arr = src.read()  # shape = (band, y, x)
    transform = src.transform
    crs = src.crs
    height = src.height
    width = src.width
    raster_dtypes = src.dtypes
    band_count = src.count
    bounds = src.bounds
    descriptions = src.descriptions

print("\nRaster info:")
print(f"  shape: {arr.shape}")
print(f"  dtypes: {raster_dtypes}")
print(f"  descriptions: {descriptions}")
print(f"  crs: {crs}")
print(f"  bounds: {bounds}")

if band_count != len(band_schema):
    raise ValueError(
        f"Band count mismatch: raster has {band_count} bands, schema has {len(band_schema)} band rows."
    )

expected_indices = list(range(1, band_count + 1))
if band_schema["band_index"].tolist() != expected_indices:
    raise ValueError(
        f"Schema band_index is not sequential 1..{band_count}: "
        f"{band_schema['band_index'].tolist()}"
    )


# ============================================================
# BUILD COORDINATES
# ============================================================
x_coords = transform.c + (np.arange(width) + 0.5) * transform.a
y_coords = transform.f + (np.arange(height) + 0.5) * transform.e


# ============================================================
# BUILD XARRAY DATASET
# ============================================================
data_vars = {}
encoding = {}

for _, row in band_schema.iterrows():
    band_idx0 = int(row["band_index"]) - 1
    band_name = str(row["band_name"])
    desc = None if pd.isna(row["description"]) else str(row["description"])
    units = None if pd.isna(row["units"]) else str(row["units"])
    nodata = maybe_numeric(row["nodata"])
    out_dtype = dtype_from_schema(row["dtype"])

    band_data = arr[band_idx0].astype(np.int16, copy=False)

    da = xr.DataArray(
        band_data,
        dims=("y", "x"),
        coords={"y": y_coords, "x": x_coords},
        name=band_name,
        attrs={
            "long_name": band_name,
            "description": desc,
            "units": units,
            "nodata": nodata,
            "source_band_index": band_idx0 + 1,
            "source_dtype": str(raster_dtypes[band_idx0]),
            "schema_dtype": str(row["dtype"]),
        },
    )

    data_vars[band_name] = da

    enc = {
        "zlib": True,
        "complevel": 4,
        "shuffle": True,
        "dtype": out_dtype,
    }
    if nodata is not None:
        enc["_FillValue"] = np.int16(nodata)
    encoding[band_name] = enc

ds = xr.Dataset(data_vars=data_vars)

ds.attrs.update({
    "title": "MTBS burn context dataset",
    "source_tif": str(tif_path),
    "source_schema_csv": str(schema_path),
    "crs": str(crs) if crs is not None else "",
    "transform": tuple(transform),
    "width": width,
    "height": height,
    "bounds_left": bounds.left,
    "bounds_bottom": bounds.bottom,
    "bounds_right": bounds.right,
    "bounds_top": bounds.top,
})


# ============================================================
# VALIDATION
# ============================================================
expected_bands = [
    "mtbs_most_recent_burn_year",
    "mtbs_time_since_most_recent_burn",
    "mtbs_burn_count",
    "mtbs_ever_burned",
    "mtbs_most_recent_severity",
]

missing = [b for b in expected_bands if b not in ds.data_vars]
extra = [b for b in ds.data_vars if b not in expected_bands]

print("\nValidation:")
print("  Missing expected bands:", missing if missing else "None")
print("  Extra bands:", extra if extra else "None")

for band in ds.data_vars:
    vals = ds[band].values
    nodata = ds[band].attrs.get("nodata", None)

    if nodata is not None:
        valid = vals != nodata
    else:
        valid = np.ones(vals.shape, dtype=bool)

    if valid.any():
        vmin = vals[valid].min()
        vmax = vals[valid].max()
    else:
        vmin = None
        vmax = None

    print(f"{band}: min={vmin}, max={vmax}, nodata={nodata}")

print("\nDataset summary:")
print(ds)


# ============================================================
# WRITE FINAL NETCDF
# ============================================================
ds.to_netcdf(out_nc, mode="w", format="NETCDF4", encoding=encoding)

print(f"\nWrote finalized dataset to:\n{out_nc}")


Schema columns:
['system:index', 'band_index', 'band_name', 'description', 'dtype', 'nodata', 'units', '.geo']

Band schema:
   band_index                         band_name  dtype  \
0           1        mtbs_most_recent_burn_year  int16   
1           2  mtbs_time_since_most_recent_burn  int16   
2           3                   mtbs_burn_count  int16   
3           4                  mtbs_ever_burned  int16   
4           5         mtbs_most_recent_severity  int16   

                      units  nodata  
0                      year  -32768  
1                     years  -32768  
2                     count       0  
3                    binary       0  
4  MTBS_Burn_Severity_Class     255  

Raster info:
  shape: (5, 7857, 10435)
  dtypes: ('int16', 'int16', 'int16', 'int16', 'int16')
  descriptions: ('mtbs_most_recent_burn_year', 'mtbs_time_since_most_recent_burn', 'mtbs_burn_count', 'mtbs_ever_burned', 'mtbs_most_recent_severity')
  crs: EPSG:26911
  bounds: BoundingBox(left=52774

In [2]:
import rasterio
import numpy as np

tif_path = r"D:\My Drive\GEE_exports\MTBS_BurnContext_Multiband_10m.tif"

with rasterio.open(tif_path) as src:
    print("=== BASIC INFO ===")
    print("Band count:", src.count)
    print("Shape (rows, cols):", (src.height, src.width))
    print("CRS:", src.crs)
    print("Transform:", src.transform)
    print("Dtypes:", src.dtypes)
    
    print("\n=== BAND NAMES (if present) ===")
    print(src.descriptions)  # often None for GEE exports

    print("\n=== PER-BAND SUMMARY ===")
    for i in range(1, src.count + 1):
        data = src.read(i)
        
        # basic stats
        finite = np.isfinite(data)
        if finite.any():
            vmin = data[finite].min()
            vmax = data[finite].max()
        else:
            vmin, vmax = None, None

        unique_sample = np.unique(data[::100, ::100])  # sample to avoid huge compute

        print(f"\nBand {i}")
        print("  dtype:", data.dtype)
        print("  min/max:", vmin, vmax)
        print("  unique (sample):", unique_sample[:10])

=== BASIC INFO ===
Band count: 5
Shape (rows, cols): (7857, 10435)
CRS: EPSG:26911
Transform: | 10.00, 0.00, 527740.00|
| 0.00,-10.00, 4815410.00|
| 0.00, 0.00, 1.00|
Dtypes: ('int16', 'int16', 'int16', 'int16', 'int16')

=== BAND NAMES (if present) ===
('mtbs_most_recent_burn_year', 'mtbs_time_since_most_recent_burn', 'mtbs_burn_count', 'mtbs_ever_burned', 'mtbs_most_recent_severity')

=== PER-BAND SUMMARY ===

Band 1
  dtype: int16
  min/max: -32768 2023
  unique (sample): [-32768      0   1984   1985   1986   1987   1989   1991   1993   1994]

Band 2
  dtype: int16
  min/max: -32768 41
  unique (sample): [-32768      0      2      3      5      7      8     10     11     12]

Band 3
  dtype: int16
  min/max: 0 6
  unique (sample): [0 1 2 3 4 5]

Band 4
  dtype: int16
  min/max: 0 1
  unique (sample): [0 1]

Band 5
  dtype: int16
  min/max: 0 255
  unique (sample): [  0   1   2   3   4   5   6 255]


In [5]:
import rasterio
import numpy as np

tif_path = r"D:\My Drive\GEE_exports\MTBS_BurnContext_Multiband_10m_finalized.nc"

with rasterio.open(tif_path) as src:
    print("=== BASIC INFO ===")
    print("Band count:", src.count)
    print("Shape (rows, cols):", (src.height, src.width))
    print("CRS:", src.crs)
    print("Transform:", src.transform)
    print("Dtypes:", src.dtypes)
    
    print("\n=== BAND NAMES (if present) ===")
    print(src.descriptions)  # often None for GEE exports

    print("\n=== PER-BAND SUMMARY ===")
    for i in range(1, src.count + 1):
        data = src.read(i)
        
        # basic stats
        finite = np.isfinite(data)
        if finite.any():
            vmin = data[finite].min()
            vmax = data[finite].max()
        else:
            vmin, vmax = None, None

        unique_sample = np.unique(data[::100, ::100])  # sample to avoid huge compute

        print(f"\nBand {i}")
        print("  dtype:", data.dtype)
        print("  min/max:", vmin, vmax)
        print("  unique (sample):", unique_sample[:10])

=== BASIC INFO ===
Band count: 0
Shape (rows, cols): (512, 512)
CRS: None
Transform: | 1.00, 0.00, 0.00|
| 0.00, 1.00, 0.00|
| 0.00, 0.00, 1.00|
Dtypes: ()

=== BAND NAMES (if present) ===
()

=== PER-BAND SUMMARY ===


c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\rasterio\__init__.py:356: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)


variable labels below

In [ ]:
MTBS_CATEGORICAL_LABELS = {
    "mtbs_ever_burned": {
        0: "not_burned",
        1: "burned",
    },
    "mtbs_most_recent_severity": {
        0: "background",
        1: "unburned_to_low",
        2: "low",
        3: "moderate",
        4: "high",
        5: "increased_greenness",
        6: "non_mapping_area",
        255: "nodata",  # your pipeline fill value, not MTBS native class
    },
}

MTBS_CONTINUOUS_OR_COUNT_VARS = {
    "mtbs_most_recent_burn_year": {
        "nodata": -32768,
        "units": "year",
    },
    "mtbs_time_since_most_recent_burn": {
        "nodata": -32768,
        "units": "years",
    },
    "mtbs_burn_count": {
        "nodata": None,  # 0 is valid
        "units": "count",
    },
}